In [1]:
# Colab Spark Setup: Single Cell

# 1. Install PySpark (includes Spark + Hadoop + Py4J)
!pip install --quiet pyspark==3.5.1

# 2. Import Spark and create session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ColabSparkSetup") \
    .getOrCreate()



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 15.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

# 1. Initialize Spark
spark = SparkSession.builder.appName("PrepareFlightDelayTestData").getOrCreate()

# 2. Load raw test CSV
input_path = "/content/drive/MyDrive/Project/combined_2024_final.csv"  # change this to your path
df_test = spark.read.csv(input_path, header=True, inferSchema=True)

print("Original Test Data Schema:")
df_test.printSchema()

# 3. Rename columns to match training
df_test = (
    df_test
    .withColumnRenamed("origin", "ORIGIN")
    .withColumnRenamed("destination", "DEST")
    .withColumnRenamed("CRSDepTime", "CRS_DEP_TIME")
    .withColumnRenamed("Month", "MONTH")
    .withColumnRenamed("DayOfWeek", "DAY_OF_WEEK")
    .withColumnRenamed("weather_type", "Type")
    .withColumnRenamed("severity", "Severity")
    .withColumnRenamed("precipitation", "Precipitation(in)")
)

# 4. Create missing features

# HOUR_OF_DAY from CRS_DEP_TIME
df_test = df_test.withColumn("HOUR_OF_DAY", (col("CRS_DEP_TIME")/100).cast("int"))

# SEASON using month mapping
df_test = df_test.withColumn(
    "SEASON",
    when(col("MONTH").isin(12,1,2), "Winter")
    .when(col("MONTH").isin(3,4,5), "Spring")
    .when(col("MONTH").isin(6,7,8), "Summer")
    .otherwise("Fall")
)

# SEVERITY_SCORE mapping
df_test = df_test.withColumn(
    "SEVERITY_SCORE",
    when(col("Severity")=="Light",1)
    .when(col("Severity")=="Moderate",2)
    .when(col("Severity")=="Heavy",3)
    .when(col("Severity")=="Severe",4)
    .otherwise(1)
)

# PRECIP_CAT mapping
df_test = df_test.withColumn(
    "PRECIP_CAT",
    when(col("Precipitation(in)") <= 0.01, "None")
    .when(col("Precipitation(in)") <= 0.1, "Light")
    .when(col("Precipitation(in)") <= 0.5, "Moderate")
    .otherwise("Heavy")
)

# 5. Select the 12 model features
selected_features = [
    'ORIGIN','DEST','CRS_DEP_TIME','MONTH','DAY_OF_WEEK','HOUR_OF_DAY',
    'SEASON','Type','Severity','SEVERITY_SCORE','Precipitation(in)','PRECIP_CAT'
]

df_final = df_test.select(*selected_features)

# 6. Save final prepared CSV for model prediction
output_path = "/content/drive/MyDrive/Project/prepared_test_data.csv"
df_final.toPandas().to_csv(output_path, index=False)

print(f"✅ Prepared test data saved to {output_path}")

spark.stop()


Original Test Data Schema:
root
 |-- is_delay: double (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- DestState: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- Cancelled: double (nullable = true)
 |-- Diverted: double (nullable = true)
 |-- Distance: double (nullable = true)
 |-- DistanceGroup: integer (nullable = true)
 |-- ArrDelay: double (nullable = true)
 |-- ArrDelayMinutes: double (nullable = true)
 |-- AirTime: double (nullable = true)
 |-- EventId: string (nullable = true)
 |-- weather_type: string (nullable = true)
 |-- severity: string (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- TimeZone: string (nullable = true)
 |-- Ai

In [ ]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 3.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import category_encoders as ce

# 1. Load your training and test data
train_df = pd.read_csv("training_data.csv")  # your training dataset
test_df = pd.read_csv("/content/drive/MyDrive/Project/prepared_test_data.csv")  # after feature selection

# 2. Define target and categorical columns
y_train = train_df['DELAY_DUE_WEATHER_YN'].map({'Yes': 1, 'No': 0})
categorical_cols = ['ORIGIN', 'DEST', 'Type', 'Severity', 'SEASON', 'TIME_CATEGORY', 'PRECIP_CAT']

# 3. Fit TargetEncoder on training data
encoder = ce.TargetEncoder(cols=categorical_cols)
encoder.fit(train_df[categorical_cols], y_train)

# 4. Transform test data
X_test_encoded = test_df.copy()
X_test_encoded[categorical_cols] = encoder.transform(test_df[categorical_cols])

# 5. Save the encoded test data
X_test_encoded.to_csv("test_encoded.csv", index=False)
print("✅ Test data encoded and saved as test_encoded.csv")


NameError: name 'encoder' is not defined

In [ ]:
# =========================
# 1. Import Libraries
# =========================
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, when
from pyspark.sql.types import IntegerType, StringType
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Start Spark session
spark = SparkSession.builder.appName("FlightDelayTestProcessing").getOrCreate()

# =========================
# 2. Load Test Dataset
# =========================
input_csv = "/content/drive/MyDrive/Project/combined_2024_final.csv"  # Change to your actual test CSV
df = spark.read.csv(input_csv, header=True, inferSchema=True)

print("Original columns in test data:")
print(df.columns)

# =========================
# 3. Select Required Columns & Rename
# =========================
# Your test dataset columns
# ['is_delay','Quarter','Month','DayofMonth','DayOfWeek','Reporting_Airline','origin','OriginState','destination','DestState','CRSDepTime',
#  'Cancelled','Diverted','Distance','DistanceGroup','ArrDelay','ArrDelayMinutes','AirTime','EventId','weather_type','severity',
#  'precipitation','TimeZone','AirportCode','LocationLat','LocationLng','City','County','State','ZipCode','IATA','date','StartTime(UTC)','EndTime(UTC)']

# Rename to match training feature names
df = df.withColumnRenamed("origin", "ORIGIN") \
       .withColumnRenamed("destination", "DEST") \
       .withColumnRenamed("CRSDepTime", "CRS_DEP_TIME") \
       .withColumnRenamed("Month", "MONTH") \
       .withColumnRenamed("DayOfWeek", "DAY_OF_WEEK") \
       .withColumnRenamed("weather_type", "Type") \
       .withColumnRenamed("severity", "Severity") \
       .withColumnRenamed("precipitation", "Precipitation(in)") \
       .withColumnRenamed("date", "FL_DATE")

# =========================
# 4. Feature Engineering (Like Training)
# =========================

# --- Hour of day from CRS_DEP_TIME ---
df = df.withColumn("HOUR_OF_DAY", (col("CRS_DEP_TIME")/100).cast(IntegerType()))

# --- Season Feature ---
def get_season(month):
    if month in [12, 1, 2]: return "Winter"
    elif month in [3, 4, 5]: return "Spring"
    elif month in [6, 7, 8]: return "Summer"
    else: return "Fall"

season_udf = udf(get_season, StringType())
df = df.withColumn("SEASON", season_udf(col("MONTH")))

# --- Precipitation Category ---
def get_precip_cat(p):
    if p is None: return "None"
    if p == 0: return "None"
    elif p < 0.1: return "Light"
    elif p < 0.5: return "Moderate"
    else: return "Heavy"

precip_udf = udf(get_precip_cat, StringType())
df = df.withColumn("PRECIP_CAT", precip_udf(col("Precipitation(in)")))

# --- Severity Score ---
def get_severity_score(s):
    mapping = {
        "Light": 1,
        "Moderate": 2,
        "Heavy": 3,
        "Severe": 4,
        "Unknown": 1
    }
    return mapping.get(s, 1)

severity_udf = udf(get_severity_score, IntegerType())
df = df.withColumn("SEVERITY_SCORE", severity_udf(col("Severity")))

# =========================
# 5. Select Only the 12 Features for Model
# =========================
selected_features = [
    'FL_DATE','ORIGIN', 'DEST', 'CRS_DEP_TIME', 'MONTH', 'DAY_OF_WEEK',
    'HOUR_OF_DAY', 'SEASON', 'Type', 'Severity',
    'SEVERITY_SCORE', 'Precipitation(in)', 'PRECIP_CAT'
]

df_features = df.select(selected_features)

# Convert to Pandas for Label Encoding
pdf = df_features.toPandas()

# =========================

# =========================
# 7. Save Processed Test Data
# =========================
output_csv = "/content/drive/MyDrive/Project/processed_test_data.csv"
pdf.to_csv(output_csv, index=False)
print(f"\nProcessed test data saved as: {output_csv}")

# =========================
# Now you can load this CSV for prediction with your XGBoost model
# =========================


Original columns in test data:
['is_delay', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'Reporting_Airline', 'origin', 'OriginState', 'destination', 'DestState', 'CRSDepTime', 'Cancelled', 'Diverted', 'Distance', 'DistanceGroup', 'ArrDelay', 'ArrDelayMinutes', 'AirTime', 'EventId', 'weather_type', 'severity', 'precipitation', 'TimeZone', 'AirportCode', 'LocationLat', 'LocationLng', 'City', 'County', 'State', 'ZipCode', 'IATA', 'date', 'StartTime(UTC)', 'EndTime(UTC)']

Processed test data saved as: /content/drive/MyDrive/Project/processed_test_data.csv
